In [7]:
# ============================================================================
# ADIM 1: RESNET50 İLE NİHAİ DÜZELTİLMİŞ EĞİTİM (Dengesizlik Giderme - Oversampling)
# ============================================================================

# Ensure TensorFlow is installed in this notebook environment (fixes "could not be resolved")
# Note: magic must be at top of cell for notebook installs
%pip install -q "tensorflow>=2.9.0"

import os
import numpy as np
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils import resample # Yeniden örnekleme için
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import warnings

# ResNet50 için özel pre-processing fonksiyonu
# Some environments may not expose resnet_v2; add a safe fallback to resnet50 preprocess_input
try:
    from tensorflow.keras.applications.resnet_v2 import preprocess_input as resnet_preprocess
except Exception:
    from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

warnings.filterwarnings('ignore')

# --- TEMEL AYARLAR ---
IMAGE_WIDTH, IMAGE_HEIGHT = 224, 224
BATCH_SIZE = 32
INITIAL_EPOCHS = 10
FINE_TUNE_EPOCHS = 20
NUM_CLASSES = 3
MODEL_SAVE_PATH = 'models/busi_cancer_ft_resnet_final_balanced.h5' # Dengeleme yapıldığı belirtildi
INITIAL_LR = 0.0001     
FINE_TUNE_LR = 0.0000005 
MEDIAN_FILTER_SIZE = 3 

# --- VERİ YOLLARI ---
BASE_DIR = r'C:\Users\kemal\Desktop\Proje\dataset' 

BUSI_CLASS_MAP = {
    'normal': (0, os.path.join('no', 'normal')), 
    'benign': (1, os.path.join('yes', 'benign')),
    'malignant': (2, os.path.join('yes', 'malignant'))
}

# --- VERİ YÜKLEME VE MEDIAN FILTER UYGULAMASI ---
def load_data_from_custom_paths(base_dir, class_map, median_filter_size):
    all_images, all_labels = [], []
    print("Veri Yükleme Başladı...")
    
    for class_name, (label_id, relative_path) in class_map.items():
        folder_path = os.path.join(base_dir, relative_path)
        
        if not os.path.isdir(folder_path):
            print(f"❌ KRİTİK HATA: Klasör bulunamadı: {folder_path}")
            continue

        count = 0
        for filename in os.listdir(folder_path):
            if 'mask' not in filename.lower() and filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(folder_path, filename)
                try:
                    img = cv2.imdecode(np.fromfile(img_path, dtype=np.uint8), cv2.IMREAD_COLOR)
                    
                    if img is None: continue
                    
                    # Gürültü Giderme (Median Filter)
                    img = cv2.medianBlur(img, median_filter_size) 
                    
                    # Ön işleme
                    if len(img.shape) == 2:
                        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    
                    img = cv2.resize(img, (IMAGE_WIDTH, IMAGE_HEIGHT)).astype(np.float32)
                    
                    all_images.append(img)
                    all_labels.append(label_id)
                    count += 1
                except Exception:
                    pass
        print(f"✅ Yüklendi: {class_name} ({count} görüntü)")

    if not all_images:
        return np.array([]), np.array([])
        
    return np.array(all_images), np.array(all_labels)

# Veri yükleme (Normalizasyon HENÜZ YOK)
all_images, all_labels = load_data_from_custom_paths(BASE_DIR, BUSI_CLASS_MAP, MEDIAN_FILTER_SIZE)

if all_images.size == 0:
    print("\n[PROGRAM SONLANDI]: Lütfen BASE_DIR yolunu kontrol edin.")
    exit()

# ImageNet Ön İşlemesi (Veriyi bölmeden önce yapılır)
print("⚙️ ImageNet Ön İşlemesi Uygulanıyor...")
all_images = resnet_preprocess(all_images)

# Veriyi bölme
X_train, X_test, y_train_raw, y_test_raw = train_test_split(all_images, all_labels, test_size=0.2, random_state=42, stratify=all_labels)
X_train, X_val, y_train_raw, y_val_raw = train_test_split(X_train, y_train_raw, test_size=0.15, random_state=42, stratify=y_train_raw)

# ====================================================
# AZINLIK SINIF DENGELEMESİ (OVERSAMPLING) - YENİ YÖNTEM
# ====================================================
print("\n⚖️ Eğitim Verisi Dengesizliği Düzeltiliyor (Oversampling)...")
unique_classes, counts = np.unique(y_train_raw, return_counts=True)
max_count = np.max(counts)
X_train_list, y_train_list = list(X_train), list(y_train_raw)
balanced_X_train, balanced_y_train_raw = [], []

for class_id in unique_classes:
    class_indices = np.where(y_train_raw == class_id)[0]
    X_class = X_train[class_indices]
    y_class = y_train_raw[class_indices]
    
    # Azınlık sınıfları çoğalt
    if len(X_class) < max_count:
        X_resampled, y_resampled = resample(X_class, y_class,
                                            replace=True,     # Yeniden örnekleme
                                            n_samples=max_count, # En büyük sınıf sayısına tamamla
                                            random_state=42)
    else:
        X_resampled, y_resampled = X_class, y_class
    
    balanced_X_train.append(X_resampled)
    balanced_y_train_raw.append(y_resampled)

X_train_balanced = np.concatenate(balanced_X_train)
y_train_raw_balanced = np.concatenate(balanced_y_train_raw)

# Rastgele Karıştırma (Shuffling)
shuffle_indices = np.arange(len(X_train_balanced))
np.random.shuffle(shuffle_indices)
X_train_balanced = X_train_balanced[shuffle_indices]
y_train_raw_balanced = y_train_raw_balanced[shuffle_indices]

# Sınıf Ağırlıkları (Hala Koruyoruz, Çift Koruma)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_raw_balanced), y=y_train_raw_balanced)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

# One-hot encoding
y_train = tf.keras.utils.to_categorical(y_train_raw_balanced, NUM_CLASSES)
y_val = tf.keras.utils.to_categorical(y_val_raw, NUM_CLASSES)
y_test = tf.keras.utils.to_categorical(y_test_raw, NUM_CLASSES)

print(f"✅ Yeni Denge: Train: {len(X_train_balanced)} ({len(X_train_balanced) - len(X_train)} örnek eklendi)")
print(f"⚖️ Sınıf Ağırlıkları (Çift Koruma): {class_weight_dict}")


# --- MODEL OLUŞTURMA VE EĞİTİM ---
def create_transfer_model(input_shape, lr, num_classes):
    base_model = tf.keras.applications.ResNet50(input_shape=input_shape, include_top=False, weights='imagenet')
    
    for layer in base_model.layers: layer.trainable = False 

    x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
    x = tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.4)(x) 
    predictions = tf.keras.layers.Dense(num_classes, activation='softmax')(x) 

    model = tf.keras.models.Model(inputs=base_model.input, outputs=predictions)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# EĞİTİM İÇİN HAZIRLIK
model = create_transfer_model(input_shape=(IMAGE_WIDTH, IMAGE_HEIGHT, 3), lr=INITIAL_LR, num_classes=NUM_CLASSES) 
os.makedirs('models', exist_ok=True)
initial_checkpoint = tf.keras.callbacks.ModelCheckpoint('models/busi_model_initial.h5', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1, min_delta=0.0001, mode='min')
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=0.0000005, verbose=1) 

# Data Augmentation (Sadece geometrik dönüşümler kullanılır)
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=20, width_shift_range=0.15, height_shift_range=0.15,
    shear_range=0.1, zoom_range=0.2, horizontal_flip=True, 
    vertical_flip=False, brightness_range=None,
    preprocessing_function=None, 
    fill_mode='nearest'
)

# --- 1. AŞAMA: ÖN EĞİTİM ---
print("\n" + "="*70); print("1. AŞAMA: ÖN EĞİTİM BAŞLIYOR"); print("="*70)
model.fit(
    datagen.flow(X_train_balanced, y_train, batch_size=BATCH_SIZE), # YENİ DENGELİ VERİ KULLANILDI
    steps_per_epoch=len(X_train_balanced) // BATCH_SIZE, # YENİ STEPS PER EPOCH
    epochs=INITIAL_EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=[initial_checkpoint, early_stopping, reduce_lr],
    class_weight=class_weight_dict,
    verbose=1 
)

# --- 2. AŞAMA: İNCE AYAR (FINE-TUNING) ---
try:
    model = tf.keras.models.load_model('models/busi_model_initial.h5', compile=False)
except:
    print("\n❗ İLK AŞAMA MODELİ BULUNAMADI. İNCE AYAR BAŞLATILAMIYOR.")
    exit()
    
# Tüm katmanlar eğitime açılır
for layer in model.layers:
    if not isinstance(layer, (tf.keras.layers.BatchNormalization, tf.keras.layers.Dropout)):
        layer.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR), loss='categorical_crossentropy', metrics=['accuracy'])

ft_checkpoint = tf.keras.callbacks.ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
print("\n" + "="*70); print("2. AŞAMA: İNCE AYAR BAŞLIYOR"); print("="*70)

model.fit(
    datagen.flow(X_train_balanced, y_train, batch_size=BATCH_SIZE), # YENİ DENGELİ VERİ KULLANILDI
    steps_per_epoch=len(X_train_balanced) // BATCH_SIZE, # YENİ STEPS PER EPOCH
    epochs=INITIAL_EPOCHS + FINE_TUNE_EPOCHS,
    initial_epoch=INITIAL_EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=[ft_checkpoint, early_stopping, reduce_lr],
    class_weight=class_weight_dict,
    verbose=1
)

# --- DEĞERLENDİRME ---
if os.path.exists(MODEL_SAVE_PATH):
    best_model = tf.keras.models.load_model(MODEL_SAVE_PATH, compile=False)
    best_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR), loss='categorical_crossentropy', metrics=['accuracy'])

    test_loss, test_accuracy = best_model.evaluate(X_test, y_test, verbose=0)
    print(f"\n📊 Test Accuracy: {test_accuracy:.4f}")

    y_pred = best_model.predict(X_test, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)

    class_names = ['Normal', 'İyi Huylu (Benign)', 'Kötü Huylu (Malign)']
    print("\n📋 Sınıflandırma Raporu (Normal, Benign, Malign):")
    print(classification_report(y_true_classes, y_pred_classes, target_names=class_names))
else:
    print(f"\nModel '{MODEL_SAVE_PATH}' bulunamadığı için değerlendirme yapılamadı.")

Veri Yükleme Başladı...
✅ Yüklendi: normal (133 görüntü)
✅ Yüklendi: benign (437 görüntü)
✅ Yüklendi: malignant (210 görüntü)
⚙️ ImageNet Ön İşlemesi Uygulanıyor...

⚖️ Eğitim Verisi Dengesizliği Düzeltiliyor (Oversampling)...
✅ Yeni Denge: Train: 891 (361 örnek eklendi)
⚖️ Sınıf Ağırlıkları (Çift Koruma): {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0)}
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 324s 3us/step

1. AŞAMA: ÖN EĞİTİM BAŞLIYOR
Epoch 1/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4070 - loss: 1.6778
Epoch 1: val_accuracy improved from None to 0.43617, saving model to models/busi_model_initial.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 92s 3s/step - accuracy: 0.4377 - loss: 1.6281 - val_accuracy: 0.4362 - val_loss: 1.5198 - learning_rate: 1.0000e-04
Epoch 2/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 1:06 3s/step - accuracy: 0.4062 - loss: 1.7404
Epoch 2: val_accuracy did not improve from 0.43617
27/27 ━━━━━━━━━━━━━━━━━━━━ 10s 303ms/step - accuracy: 0.4062 - loss: 1.7404 - val_accuracy: 0.4362 - val_loss: 1.5168 - learning_rate: 1.0000e-04
Epoch 3/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4886 - loss: 1.4911
Epoch 3: val_accuracy did not improve from 0.43617
27/27 ━━━━━━━━━━━━━━━━━━━━ 84s 3s/step - accuracy: 0.4796 - loss: 1.4996 - val_accuracy: 0.1702 - val_loss: 1.4979 - learning_rate: 1.0000e-04
Epoch 4/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 1:19 3s/step - accuracy: 0.3438 - loss: 1.6454
Epoch 4: val_accuracy did not improve from 0.43617
27/27 ━━━━━━━━━━━━━━━━━━━━ 12s 349ms/step - accuracy: 0.3438 - loss: 1.6454 - val_accuracy: 0.1702 - val_loss: 1.4996 - learning_rate: 1.0000e-04
Epoch 5/10
27/27

27/27 ━━━━━━━━━━━━━━━━━━━━ 84s 3s/step - accuracy: 0.5367 - loss: 1.2618 - val_accuracy: 0.4574 - val_loss: 1.3326 - learning_rate: 1.0000e-04
Epoch 10/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 1:08 3s/step - accuracy: 0.5000 - loss: 1.4493
Epoch 10: val_accuracy improved from 0.45745 to 0.47872, saving model to models/busi_model_initial.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 11s 326ms/step - accuracy: 0.5000 - loss: 1.4493 - val_accuracy: 0.4787 - val_loss: 1.3272 - learning_rate: 1.0000e-04
Restoring model weights from the end of the best epoch: 10.

2. AŞAMA: İNCE AYAR BAŞLIYOR
Epoch 11/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.5164 - loss: 1.2720
Epoch 11: val_accuracy improved from None to 0.52128, saving model to models/busi_cancer_ft_resnet_final_balanced.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 268s 9s/step - accuracy: 0.5343 - loss: 1.2689 - val_accuracy: 0.5213 - val_loss: 1.3174 - learning_rate: 5.0000e-07
Epoch 12/30
 1/27 ━━━━━━━━━━━━━━━━━━━━ 3:29 8s/step - accuracy: 0.5312 - loss: 1.0782
Epoch 12: val_accuracy did not improve from 0.52128
27/27 ━━━━━━━━━━━━━━━━━━━━ 16s 299ms/step - accuracy: 0.5312 - loss: 1.0782 - val_accuracy: 0.5106 - val_loss: 1.3158 - learning_rate: 5.0000e-07
Epoch 13/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.5450 - loss: 1.2495
Epoch 13: val_accuracy did not improve from 0.52128
27/27 ━━━━━━━━━━━━━━━━━━━━ 221s 8s/step - accuracy: 0.5588 - loss: 1.2434 - val_accuracy: 0.5213 - val_loss: 1.3010 - learning_rate: 5.0000e-07
Epoch 14/30
 1/27 ━━━━━━━━━━━━━━━━━━━━ 3:09 7s/step - accuracy: 0.5938 - loss: 1.3179
Epoch 14: val_accuracy did not improve from 0.52128
27/27 ━━━━━━━━━━━━━━━━━━━━ 15s 288ms/step - accuracy: 0.5938 - loss: 1.3179 - val_accuracy: 0.5213 - val_loss: 1.3002 - learning_rate: 5.0000e-07
Epoch 15

27/27 ━━━━━━━━━━━━━━━━━━━━ 205s 8s/step - accuracy: 0.5763 - loss: 1.2401 - val_accuracy: 0.5319 - val_loss: 1.2818 - learning_rate: 5.0000e-07
Epoch 18/30
 1/27 ━━━━━━━━━━━━━━━━━━━━ 3:13 7s/step - accuracy: 0.5938 - loss: 1.2597
Epoch 18: val_accuracy did not improve from 0.53191
27/27 ━━━━━━━━━━━━━━━━━━━━ 15s 280ms/step - accuracy: 0.5938 - loss: 1.2597 - val_accuracy: 0.5319 - val_loss: 1.2828 - learning_rate: 5.0000e-07
Epoch 19/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.5998 - loss: 1.1953
Epoch 19: val_accuracy did not improve from 0.53191
27/27 ━━━━━━━━━━━━━━━━━━━━ 217s 8s/step - accuracy: 0.6088 - loss: 1.1687 - val_accuracy: 0.5213 - val_loss: 1.2606 - learning_rate: 5.0000e-07
Epoch 20/30
 1/27 ━━━━━━━━━━━━━━━━━━━━ 3:19 8s/step - accuracy: 0.4375 - loss: 1.4792
Epoch 20: val_accuracy did not improve from 0.53191
27/27 ━━━━━━━━━━━━━━━━━━━━ 15s 293ms/step - accuracy: 0.4375 - loss: 1.4792 - val_accuracy: 0.5000 - val_loss: 1.2604 - learning_rate: 5.0000e-07
Epoch 21

27/27 ━━━━━━━━━━━━━━━━━━━━ 216s 8s/step - accuracy: 0.6042 - loss: 1.1884 - val_accuracy: 0.6064 - val_loss: 1.2103 - learning_rate: 5.0000e-07
Epoch 22/30
 1/27 ━━━━━━━━━━━━━━━━━━━━ 3:14 7s/step - accuracy: 0.7500 - loss: 0.8676
Epoch 22: val_accuracy did not improve from 0.60638
27/27 ━━━━━━━━━━━━━━━━━━━━ 15s 303ms/step - accuracy: 0.7500 - loss: 0.8676 - val_accuracy: 0.6064 - val_loss: 1.2078 - learning_rate: 5.0000e-07
Epoch 23/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.5980 - loss: 1.1775
Epoch 23: val_accuracy did not improve from 0.60638
27/27 ━━━━━━━━━━━━━━━━━━━━ 208s 8s/step - accuracy: 0.5809 - loss: 1.2185 - val_accuracy: 0.5319 - val_loss: 1.2417 - learning_rate: 5.0000e-07
Epoch 24/30
 1/27 ━━━━━━━━━━━━━━━━━━━━ 3:05 7s/step - accuracy: 0.6875 - loss: 1.0916
Epoch 24: val_accuracy did not improve from 0.60638
27/27 ━━━━━━━━━━━━━━━━━━━━ 14s 280ms/step - accuracy: 0.6875 - loss: 1.0916 - val_accuracy: 0.5319 - val_loss: 1.2395 - learning_rate: 5.0000e-07
Epoch 25

27/27 ━━━━━━━━━━━━━━━━━━━━ 210s 8s/step - accuracy: 0.6356 - loss: 1.0992 - val_accuracy: 0.6596 - val_loss: 1.1077 - learning_rate: 5.0000e-07
Epoch 30/30
 1/27 ━━━━━━━━━━━━━━━━━━━━ 3:12 7s/step - accuracy: 0.6875 - loss: 1.0458
Epoch 30: val_accuracy did not improve from 0.65957
27/27 ━━━━━━━━━━━━━━━━━━━━ 15s 288ms/step - accuracy: 0.6875 - loss: 1.0458 - val_accuracy: 0.6596 - val_loss: 1.1111 - learning_rate: 5.0000e-07
Restoring model weights from the end of the best epoch: 29.

📊 Test Accuracy: 0.7179

📋 Sınıflandırma Raporu (Normal, Benign, Malign):
                     precision    recall  f1-score   support

             Normal       0.53      0.63      0.58        27
 İyi Huylu (Benign)       0.78      0.82      0.80        87
Kötü Huylu (Malign)       0.73      0.57      0.64        42

           accuracy                           0.72       156
          macro avg       0.68      0.67      0.67       156
       weighted avg       0.72      0.72      0.72       156



In [25]:
# ============================================================================
# ADIM 2: GUI UYGULAMASI (MODERN GÖRÜNÜM VE OPTİMİZE FONKSİYONLAR)
# ============================================================================

# Uyarıları bastırmak için gerekli importlar
import warnings
import os
import sys

# TensorFlow/Keras'ın HDF5 uyarısını bastır (Sarı çizgiye neden olan ana sorun)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
warnings.filterwarnings('ignore')

import tkinter as tk
from tkinter import filedialog, Label, Frame
import cv2
import numpy as np
import tensorflow as tf
from PIL import Image, ImageTk

# ADIM 1'de tanımlanan ResNet ön işleme fonksiyonuna erişimi garantileme
# Bu kısım, ADIM 1'deki kodunuz ile aynı ortamda çalışmayı sağlar.
try:
    from tensorflow.keras.applications.resnet_v2 import preprocess_input as resnet_preprocess
except Exception:
    from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

# --- UYGULAMA AYARLARI ---
MODEL_PATH = 'models/busi_cancer_ft_resnet_final_balanced.h5'
IMAGE_WIDTH, IMAGE_HEIGHT = 224, 224
WINDOW_WIDTH, WINDOW_HEIGHT = 900, 700
class_names = ['Normal', 'İyi Huylu (Benign)', 'Kötü Huylu (Malign)']
MEDIAN_FILTER_SIZE = 3 # ADIM 1'deki ayar ile uyumlu olmalı

# RENKLER (Modern Koyu Tema)
BG_DARK = "#1E1E1E" 
BG_MID = "#2D2D2D"  
ACCENT_GREEN = "#4CAF50" 
ACCENT_BLUE = "#007ACC"  
ACCENT_RED = "#D32F2F"   
ACCENT_ORANGE = "#FF9800"
FONT_STYLE = ("Segoe UI", 12)

# Global değişkenler
cap = None
is_camera_running = False
model = None
app_initialized = False

# --- 1. MODELİ GÜVENLİ ŞEKİLDE YÜKLEME ---
if os.path.exists(MODEL_PATH):
    try:
        # Uyarıyı önlemek için compile=False kullanmak en iyisidir.
        model = tf.keras.models.load_model(MODEL_PATH, compile=False)
        app_initialized = True
    except Exception as e:
        print(f"HATA: Model yüklenirken bir sorun oluştu: {e}")
        pass
else:
    print(f"HATA: Model dosyası '{MODEL_PATH}' bulunamadı. Lütfen ADIM 1'i tamamlayın.")

# --- 2. GÖRÜNTÜ VE TAHMİN FONKSİYONLARI ---
def predict_on_image(image):
    if model is None: return "HATA: Model yüklenemedi.", "gray"
    
    # Görüntüyü BGR'den RGB'ye dönüştürme (cv2 BGR okur)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # 1. Medyan Filtre Uygulama (Gürültü Giderme) - Eğitimdeki Adım 
    image_filtered = cv2.medianBlur(image_rgb, MEDIAN_FILTER_SIZE)
    
    # 2. Yeniden Boyutlandırma
    image_resized = cv2.resize(image_filtered, (IMAGE_WIDTH, IMAGE_HEIGHT))
    
    # 3. ResNet Ön İşlemesi (ImageNet Normalizasyonu) - Eğitimdeki Kritik Adım
    image_for_preprocess = image_resized.astype(np.float32)
    image_for_preprocess = np.expand_dims(image_for_preprocess, axis=0) 
    
    # ResNet'in özel normalizasyonunu uygula
    image_for_prediction = resnet_preprocess(image_for_preprocess)
    
    # Tahmin yap
    # verbose=0 uyarı/çıktı kirliliğini azaltır
    predictions = model.predict(image_for_prediction, verbose=0)[0] 
    
    prediction_index = np.argmax(predictions)
    confidence = predictions[prediction_index] * 100
    predicted_class = class_names[prediction_index]
    
    # Renk ve Durum Atama
    if predicted_class == 'Kötü Huylu (Malign)':
        status = "!!! YÜKSEK KANSER RİSKİ !!!"
        color = ACCENT_RED
    elif predicted_class == 'İyi Huylu (Benign)':
        status = "Kist Tespit Edildi (İyi Huylu)"
        color = ACCENT_ORANGE
    else:
        status = "Normal / Sağlıklı Dokular"
        color = ACCENT_GREEN
        
    return f"{status}\nTahmin: {predicted_class} ({confidence:.2f}%)", color

def update_camera_feed():
    global is_camera_running, cap
    if not is_camera_running or cap is None: return
    
    ret, frame = cap.read()
    if ret:
        label, color = predict_on_image(frame)
        prediction_label.config(text=label, fg=color)
        
        # Görüntüyü arayüze göstermek için
        img_rgb_display = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img_pil = Image.fromarray(img_rgb_display).resize((500, 350))
        img_tk = ImageTk.PhotoImage(image=img_pil)
        
        image_label.imgtk = img_tk
        image_label.config(image=img_tk, bg=BG_MID)
        image_label.after(15, update_camera_feed)
    else:
        stop_camera()

def start_live_camera():
    global cap, is_camera_running
    if not app_initialized: 
        prediction_label.config(text="HATA: Model Yüklenmedi. Eğitimi tamamlayın.", fg=ACCENT_RED)
        return
    if is_camera_running: return
    
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        prediction_label.config(text="HATA: Kamera başlatılamadı (0)", fg=ACCENT_RED)
        return
    
    is_camera_running = True
    stop_button.config(state=tk.NORMAL)
    update_camera_feed()

def stop_camera():
    global cap, is_camera_running
    if cap:
        cap.release()
        cap = None
    is_camera_running = False
    
    if 'stop_button' in globals():
        stop_button.config(state=tk.DISABLED)
    if 'prediction_label' in globals():
        prediction_label.config(text="Kamera durduruldu.", fg="white")

def load_and_predict_image():
    stop_camera()
    if not app_initialized: return

    file_path = filedialog.askopenfilename(
        title="Bir Ultrason Görüntüsü Seçin",
        filetypes=[("Image Files", "*.jpg *.jpeg *.png")]
    )
    if not file_path: return
    
    # Güvenli Görüntü Yükleme
    original_image = cv2.imdecode(np.fromfile(file_path, dtype=np.uint8), cv2.IMREAD_COLOR)
    
    if original_image is None:
        prediction_label.config(text=f"HATA: Görüntü okunamadı! {os.path.basename(file_path)}", fg=ACCENT_RED)
        return
    
    label, color = predict_on_image(original_image)
    prediction_label.config(text=label, fg=color)
    
    # Görüntüyü arayüzde göstermek için dönüştürme
    img_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
    
    # Boyutlandırma
    h, w, _ = img_rgb.shape
    scale = min(500/w, 350/h)
    new_w, new_h = int(w*scale), int(h*scale)
    
    img_pil = Image.fromarray(img_rgb).resize((new_w, new_h))
    img_tk = ImageTk.PhotoImage(image=img_pil)
    
    # Arayüzdeki Label'ı güncelle
    image_label.imgtk = img_tk
    image_label.config(image=img_tk, width=new_w, height=new_h, bg=BG_MID)

# --- 3. ANA ARAYÜZ (GUI) BAŞLATICI ---
if app_initialized:
    root = tk.Tk()
    root.title("MEME KANSERİ TESPİTİ YAPAY ZEKA SİSTEMİ")
    root.geometry(f"{WINDOW_WIDTH}x{WINDOW_HEIGHT}")
    root.configure(bg=BG_DARK)
    
    top_frame = Frame(root, bg=BG_DARK)
    top_frame.pack(pady=15)
    
    button_config = {
        'font': ("Segoe UI", 11, "bold"),
        'width': 22,
        'fg': "white",
        'relief': tk.FLAT, 
        'bd': 0,
        'activebackground': "#1C1C1C",
        'activeforeground': "white"
    }
    
    camera_button = tk.Button(top_frame, text="▶️ Canlı Kamera Başlat", command=start_live_camera, bg=ACCENT_GREEN, **button_config)
    camera_button.pack(side=tk.LEFT, padx=10)
    
    stop_button = tk.Button(top_frame, text="◼️ Kamerayı Durdur", command=stop_camera, bg=ACCENT_RED, **button_config, state=tk.DISABLED)
    stop_button.pack(side=tk.LEFT, padx=10)
    
    file_button = tk.Button(top_frame, text="📂 Dosyadan Görüntü Yükle", command=load_and_predict_image, bg=ACCENT_BLUE, **button_config)
    file_button.pack(side=tk.LEFT, padx=10)
    
    image_label = Label(root, bg=BG_MID, width=500, height=350, bd=0, relief=tk.FLAT)
    image_label.pack(pady=20, padx=20, expand=True)
    
    prediction_label = Label(root, text="Lütfen bir ultrason görüntüsü seçin.", font=("Segoe UI", 16, "bold"), bg=BG_DARK, fg="white")
    prediction_label.pack(pady=15)
    
    def on_closing():
        stop_camera()
        root.destroy()
    
    root.protocol("WM_DELETE_WINDOW", on_closing)
    root.mainloop()